In [1]:
import sympy as sp

x = sp.symbols("x")

In [2]:
def trimf(a, b, c):
    return sp.Piecewise(
        (0, x <= a),
        ((x - a) / (b - a), x <= b),
        ((c - x) / (c - b), x < c),
        (0, True)
    )

In [3]:
def trapmf(a, b, c, d):
    if a == b:
        return sp.Piecewise(
            (0, x < a),
            (1, x <= c),
            ((d - x) / (d - c), x < d),
            (0, True)
        )

    if c == d:
        return sp.Piecewise(
            (0, x <= a),
            ((x - a) / (b - a), x < b),
            (1, x <= d),
            (0, True)
        )

    return sp.Piecewise(
        (0, x <= a),
        ((x - a) / (b - a), x < b),
        (1, x <= c),
        ((d - x) / (d - c), x < d),
        (0, True)
    )

In [4]:
def degree(expr, nilai):
    return float(expr.subs(x, nilai))

In [5]:
def input_range(teks, minimum, maksimum):
    while True:
        try:
            nilai = float(input(teks))
            if minimum <= nilai <= maksimum:
                return nilai
            print(f"Input harus dalam range {minimum}-{maksimum}.")
        except ValueError:
            print("Input harus berupa angka.")

In [6]:
def centroid(output_mf, aturan, start, end, step):
    total_titik = int((end - start) / step) + 1
    pembilang = 0.0
    penyebut = 0.0

    for i in range(total_titik):
        nilai = start + i * step
        mu = max(min(w, degree(output_mf[label], nilai)) for label, w in aturan)
        pembilang += nilai * mu
        penyebut += mu

    return pembilang / penyebut if penyebut != 0 else 0.0

In [7]:
def interpretasi_arah(nilai):
    if nilai < -45:
        return "Kiri Tajam"
    if nilai < -10:
        return "Kiri"
    if nilai <= 10:
        return "Lurus"
    if nilai <= 45:
        return "Kanan"
    return "Kanan Tajam"

In [8]:
def interpretasi_kecepatan(nilai):
    if nilai <= 35:
        return "Lambat"
    if nilai <= 70:
        return "Normal"
    return "Kencang"

In [9]:
kiri_input = input_range("Masukkan jarak sensor kiri (0-200): ", 0, 200)
kanan_input = input_range("Masukkan jarak sensor kanan (0-200): ", 0, 200)

In [10]:
jarak_mf = {
    "Dekat": trapmf(0, 0, 20, 60),
    "Sedang": trimf(40, 100, 160),
    "Jauh": trapmf(140, 180, 200, 200)
}

kecepatan_mf = {
    "Lambat": trapmf(0, 0, 15, 35),
    "Normal": trimf(25, 50, 75),
    "Kencang": trapmf(65, 85, 100, 100)
}

arah_mf = {
    "Belok_Kiri_Tajam": trapmf(-90, -90, -70, -30),
    "Belok_Kiri": trimf(-50, -25, 0),
    "Lurus": trimf(-15, 0, 15),
    "Belok_Kanan": trimf(0, 25, 50),
    "Belok_Kanan_Tajam": trapmf(30, 70, 90, 90)
}

kiri = {k: degree(v, kiri_input) for k, v in jarak_mf.items()}
kanan = {k: degree(v, kanan_input) for k, v in jarak_mf.items()}

In [11]:
rules = [
    ("R1", "Dekat", "Dekat", "Lambat", "Lurus"),
    ("R2", "Dekat", "Sedang", "Lambat", "Belok_Kanan"),
    ("R3", "Dekat", "Jauh", "Normal", "Belok_Kanan_Tajam"),
    ("R4", "Sedang", "Dekat", "Lambat", "Belok_Kiri"),
    ("R5", "Sedang", "Sedang", "Normal", "Lurus"),
    ("R6", "Sedang", "Jauh", "Normal", "Belok_Kanan"),
    ("R7", "Jauh", "Dekat", "Normal", "Belok_Kiri_Tajam"),
    ("R8", "Jauh", "Sedang", "Normal", "Belok_Kiri"),
    ("R9", "Jauh", "Jauh", "Kencang", "Lurus")
]

In [12]:
firing = []
for kode, k, r, output_kecepatan, output_arah in rules:
    w = min(kiri[k], kanan[r])
    firing.append((kode, output_kecepatan, output_arah, w))

kecepatan = centroid(kecepatan_mf, [(ok, w) for _, ok, _, w in firing], 0, 100, 0.5)
arah = centroid(arah_mf, [(oa, w) for _, _, oa, w in firing], -90, 90, 0.5)
aktif = [kode for kode, _, _, w in firing if w > 0]

In [13]:
print("INPUT:")
print(f"Jarak Kiri: {kiri_input:.2f} cm")
print(f"Jarak Kanan: {kanan_input:.2f} cm")

INPUT:
Jarak Kiri: 30.00 cm
Jarak Kanan: 150.00 cm


In [14]:
print("\n----------------------------------------")
print("FUZZIFIKASI")
print("----------------------------------------")
print(f"Sensor Kiri ({kiri_input:.0f} cm):")
print(f"Dekat: {kiri['Dekat']:.6f}")
print(f"Sedang: {kiri['Sedang']:.6f}")
print(f"Jauh: {kiri['Jauh']:.6f}")


----------------------------------------
FUZZIFIKASI
----------------------------------------
Sensor Kiri (30 cm):
Dekat: 0.750000
Sedang: 0.000000
Jauh: 0.000000


In [15]:
print(f"Sensor Kanan ({kanan_input:.0f} cm):")
print(f"Dekat: {kanan['Dekat']:.6f}")
print(f"Sedang: {kanan['Sedang']:.6f}")
print(f"Jauh: {kanan['Jauh']:.6f}")

Sensor Kanan (150 cm):
Dekat: 0.000000
Sedang: 0.166667
Jauh: 0.250000


In [16]:
print("\n----------------------------------------")
print("FIRING STRENGTH")
print("----------------------------------------")
for i in range(0, len(firing), 3):
    baris = firing[i:i + 3]
    print(" ".join(f"{kode}: {w:.6f}" for kode, _, _, w in baris))


----------------------------------------
FIRING STRENGTH
----------------------------------------
R1: 0.000000 R2: 0.166667 R3: 0.250000
R4: 0.000000 R5: 0.000000 R6: 0.000000
R7: 0.000000 R8: 0.000000 R9: 0.000000


In [17]:
print("\n----------------------------------------")
print("DEFUZZIFIKASI (Centroid)")
print("----------------------------------------")
print(f"Kecepatan: {kecepatan:.2f} cm/s")
print(f"Arah Belok: {arah:.2f} derajat")
print(f"Interpretasi: Belok {interpretasi_arah(arah)} dengan kecepatan {interpretasi_kecepatan(kecepatan)}")


----------------------------------------
DEFUZZIFIKASI (Centroid)
----------------------------------------
Kecepatan: 39.25 cm/s
Arah Belok: 50.28 derajat
Interpretasi: Belok Kanan Tajam dengan kecepatan Normal


In [18]:
print("\n----------------------------------------")
print("TOLAK UKUR VALIDASI")
print("----------------------------------------")
print(f"μ_Dekat_kiri({kiri_input:.2f}) = {kiri['Dekat']:.6f}")
print(f"μ_Sedang_kanan({kanan_input:.2f}) = {kanan['Sedang']:.6f}")
print(f"μ_Jauh_kanan({kanan_input:.2f}) = {kanan['Jauh']:.6f}")
print(f"Aturan aktif: {', '.join(aktif) if aktif else 'Tidak ada'}")
print(f"Robot belok kanan karena arah bernilai positif: {'Ya' if arah > 0 else 'Tidak'}")
print(f"Kecepatan moderate antara lambat dan normal: {'Ya' if 25 <= kecepatan <= 75 else 'Tidak'}")


----------------------------------------
TOLAK UKUR VALIDASI
----------------------------------------
μ_Dekat_kiri(30.00) = 0.750000
μ_Sedang_kanan(150.00) = 0.166667
μ_Jauh_kanan(150.00) = 0.250000
Aturan aktif: R2, R3
Robot belok kanan karena arah bernilai positif: Ya
Kecepatan moderate antara lambat dan normal: Ya
